# Week 8 — Model Evaluation and Selection

**File:** `04_model_evaluation_week8.ipynb`

This notebook loads the processed IPL dataset and every model saved during Week 7. It evaluates the models on the same held-out test set, creates comparison plots, creates confusion matrices by converting continuous performance scores into categories, checks whether ROC curves are applicable, and saves the final model-selection results.

## 1. Imports

In [ ]:
# Install missing packages only when required:
# !pip install pandas numpy matplotlib scikit-learn xgboost lightgbm joblib

import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
print("Imports completed.")

In [ ]:
from pathlib import Path

# Find the repository root whether the notebook is opened from /notebooks or the project root.
CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
if not (PROJECT_ROOT / "data" / "processed" / "ipl_features.csv").exists():
    candidates = list(CURRENT.rglob("data/processed/ipl_features.csv"))
    if not candidates:
        raise FileNotFoundError("Could not find data/processed/ipl_features.csv")
    PROJECT_ROOT = candidates[0].parents[2]

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ipl_features.csv"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
PLOTS_DIR = RESULTS_DIR / "plots"

for folder in [MODELS_DIR, RESULTS_DIR, PLOTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")

## 2. Load processed dataset and training metadata

In [ ]:
metadata_path = RESULTS_DIR / "training_metadata.json"
if not metadata_path.exists():
    raise FileNotFoundError(
        "results/training_metadata.json was not found. Run 03_model_training_week7.ipynb first."
    )

with open(metadata_path, "r", encoding="utf-8") as file:
    metadata = json.load(file)

TARGET = metadata["target"]
feature_columns = metadata["features"]
RANDOM_STATE = metadata["random_state"]
TEST_SIZE = metadata["test_size"]

df = pd.read_csv(DATA_PATH)
model_data = df[feature_columns + [TARGET]].replace([np.inf, -np.inf], np.nan)
model_data = model_data.dropna(subset=[TARGET]).reset_index(drop=True)

X = model_data[feature_columns]
y = model_data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"Dataset shape: {df.shape}")
print(f"Evaluation rows: {len(X_test)}")
print(f"Features: {len(feature_columns)}")

## 3. Load all saved models

In [ ]:
model_names = ["random_forest", "xgboost", "mlp", "hybrid", "lightgbm"]
loaded_models = {}

for name in model_names:
    model_path = MODELS_DIR / f"{name}.pkl"
    if not model_path.exists():
        raise FileNotFoundError(f"Missing {model_path}. Run the Week 7 notebook first.")
    loaded_models[name] = joblib.load(model_path)
    print(f"Loaded: {model_path.name}")

## 4. Evaluate each regression model

In [ ]:
evaluation_rows = []
predictions_by_model = {}

for name, model in loaded_models.items():
    predictions = model.predict(X_test)
    predictions_by_model[name] = predictions
    evaluation_rows.append({
        "model": name,
        "r2": r2_score(y_test, predictions),
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": mean_squared_error(y_test, predictions) ** 0.5,
    })

evaluation_df = pd.DataFrame(evaluation_rows).sort_values(
    ["rmse", "mae"], ascending=[True, True]
).reset_index(drop=True)
display(evaluation_df)

## 5. Generate confusion matrices using performance categories

In [ ]:
# Confusion matrices are classification tools. Because these models predict a continuous
# score, scores are converted into three categories using thresholds calculated only
# from the training target. This avoids using test-set information to define categories.
q1, q2 = y_train.quantile([0.33, 0.66]).tolist()
category_labels = ["Low", "Average", "High"]

def score_to_category(values):
    values = np.asarray(values)
    return np.select(
        [values <= q1, values <= q2],
        ["Low", "Average"],
        default="High",
    )

y_test_category = score_to_category(y_test)
classification_rows = []

for name, predictions in predictions_by_model.items():
    predicted_category = score_to_category(predictions)
    matrix = confusion_matrix(y_test_category, predicted_category, labels=category_labels)

    display_obj = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=category_labels,
    )
    display_obj.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix — {name.replace('_', ' ').title()}")
    plt.tight_layout()
    path = PLOTS_DIR / f"confusion_matrix_{name}.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()

    report = classification_report(
        y_test_category,
        predicted_category,
        labels=category_labels,
        output_dict=True,
        zero_division=0,
    )
    classification_rows.append({
        "model": name,
        "category_accuracy": report["accuracy"],
        "macro_f1": report["macro avg"]["f1-score"],
    })

classification_df = pd.DataFrame(classification_rows).sort_values(
    "macro_f1", ascending=False
).reset_index(drop=True)
display(classification_df)

## 6. Plot ROC curves if applicable

In [ ]:
# ROC curves normally require classification probabilities from predict_proba or
# decision_function. The saved models are regressors and return continuous performance
# scores, so a conventional ROC curve is not applicable and is intentionally not plotted.
roc_capable = {
    name: hasattr(model, "predict_proba") or hasattr(model, "decision_function")
    for name, model in loaded_models.items()
}
print("ROC capability:", roc_capable)
if not any(roc_capable.values()):
    print("ROC curves skipped: all saved models are regression models.")

## 7. Compare all models

In [ ]:
comparison_df = evaluation_df.merge(classification_df, on="model", how="left")
display(comparison_df)

comparison_df.to_csv(RESULTS_DIR / "week8_model_comparison.csv", index=False)
with open(RESULTS_DIR / "week8_model_comparison.json", "w", encoding="utf-8") as file:
    json.dump(comparison_df.to_dict(orient="records"), file, indent=2)

print("Saved final comparison metrics.")

In [ ]:
# Regression metric comparison
plot_df = comparison_df.sort_values("rmse", ascending=False)
ax = plot_df.plot(x="model", y=["rmse", "mae"], kind="bar", figsize=(10, 5))
ax.set_title("Final Regression Model Comparison")
ax.set_xlabel("Model")
ax.set_ylabel("Error")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
path = PLOTS_DIR / "week8_regression_model_comparison.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {path}")

In [ ]:
# R² comparison
plot_df = comparison_df.sort_values("r2")
ax = plot_df.plot(x="model", y="r2", kind="bar", legend=False, figsize=(9, 5))
ax.set_title("Final R² Comparison")
ax.set_xlabel("Model")
ax.set_ylabel("R² (higher is better)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
path = PLOTS_DIR / "week8_r2_comparison.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {path}")

## 8. Select and save the best model

In [ ]:
# Primary selection criterion: lowest RMSE. MAE is used as the tie-breaker.
best_row = comparison_df.sort_values(["rmse", "mae"], ascending=[True, True]).iloc[0]
best_model_name = best_row["model"]
best_model = loaded_models[best_model_name]

joblib.dump(best_model, MODELS_DIR / "best_model_week8.pkl")

final_metrics = {
    "best_model": best_model_name,
    "selection_rule": "Lowest test RMSE, with MAE as tie-breaker",
    "r2": float(best_row["r2"]),
    "mae": float(best_row["mae"]),
    "rmse": float(best_row["rmse"]),
    "category_accuracy": float(best_row["category_accuracy"]),
    "macro_f1": float(best_row["macro_f1"]),
    "category_thresholds": {"low_to_average": float(q1), "average_to_high": float(q2)},
}
with open(RESULTS_DIR / "final_model_metrics.json", "w", encoding="utf-8") as file:
    json.dump(final_metrics, file, indent=2)

print(f"Best model: {best_model_name}")
print(f"RMSE: {best_row['rmse']:.4f}")
print(f"MAE: {best_row['mae']:.4f}")
print(f"R²: {best_row['r2']:.4f}")
print("Saved models/best_model_week8.pkl")
print("Saved results/final_model_metrics.json")

## 9. Actual versus predicted plot for the selected model

In [ ]:
best_predictions = predictions_by_model[best_model_name]
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.65)
minimum = min(float(y_test.min()), float(np.min(best_predictions)))
maximum = max(float(y_test.max()), float(np.max(best_predictions)))
plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")
plt.title(f"Actual vs Predicted — {best_model_name.replace('_', ' ').title()}")
plt.xlabel("Actual Performance Score")
plt.ylabel("Predicted Performance Score")
plt.tight_layout()
path = PLOTS_DIR / "week8_best_model_actual_vs_predicted.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {path}")

## 10. Week 8 outputs

- `models/best_model_week8.pkl`
- `results/week8_model_comparison.csv`
- `results/week8_model_comparison.json`
- `results/final_model_metrics.json`
- `results/plots/confusion_matrix_*.png`
- `results/plots/week8_regression_model_comparison.png`
- `results/plots/week8_r2_comparison.png`
- `results/plots/week8_best_model_actual_vs_predicted.png`